# 04 — Random streams and NumPy memory
The production stream buffers one sweep of random sites and uniforms. A partial consumption retains unused draws, allowing identical seeded trajectories even when calls are split. The stream returns up to the requested count, not necessarily exactly that count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ising import IsingModel, Simulator
import ising.kernels as kernels
import ising.observables as obs
from ising.state import make_state
import inspect
from ising.random_stream import RandomUpdateStream

In [ ]:
N, seed = 5, 42
rng = np.random.default_rng(seed)
site_ids = rng.integers(0,N*N,size=N*N)
rows, cols = site_ids//N+1, site_ids%N+1
flat_sites = rows*(N+2)+cols
np.testing.assert_array_equal(np.ravel_multi_index((rows,cols),(N+2,N+2)),flat_sites)
print(np.column_stack((site_ids,rows,cols,flat_sites)))
print('unique sites:',len(np.unique(site_ids)),'out of',len(site_ids))

## Views versus copies
Basic slicing shares memory; advanced indexing creates a copy. `ravel()` shares memory for this contiguous padded lattice, but need not for arbitrary strides.

In [ ]:
spins=make_state(N,'minus',np.random.default_rng(seed))
interior=spins[1:-1,1:-1]; flat_spins=spins.ravel(); sampled_spins=flat_spins[flat_sites]
print(np.shares_memory(spins,interior),np.shares_memory(spins,flat_spins),np.shares_memory(spins,sampled_spins))
interior[0,0]=1
assert spins[1,1]==1
print('core strides:',interior.strides,'contiguous?',interior.flags.c_contiguous)

## Inspect buffer lifecycle
Private fields are inspected here, not mutated. `peek` is non-consuming but may refill an empty buffer. Treat returned views as read-only to avoid corrupting future draws.

In [ ]:
stream=RandomUpdateStream(N,np.random.default_rng(seed))
sites, uniforms=stream.peek(7)
np.testing.assert_array_equal(sites,flat_sites[:7])
np.testing.assert_array_equal(uniforms,rng.random(N*N)[:7])
repeated_sites,_=stream.peek(7)
np.testing.assert_array_equal(repeated_sites,sites)
assert np.shares_memory(sites,stream._sites)
old_buffer=stream._sites
stream.consume(3)
assert np.shares_memory(old_buffer,stream._sites)
remaining_sites,_=stream.peek(1000)
assert len(remaining_sites)==N*N-3
np.testing.assert_array_equal(remaining_sites,old_buffer[3:])
stream.consume(len(remaining_sites))
refilled_sites,_=stream.peek(1000)
assert len(refilled_sites)==N*N and not np.shares_memory(refilled_sites,old_buffer)
print('new buffer:',refilled_sites)

## Chunk invariance
Compare a single request with irregular chunks spanning several refills. The helper is notebook-only instrumentation, not a proposed replacement for the stream.

In [ ]:
def collect(chunks):
    stream=RandomUpdateStream(N,np.random.default_rng(seed))
    sites,uniforms=[],[]
    for count in chunks:
        while count:
            a,b=stream.peek(count)
            sites.append(a.copy()); uniforms.append(b.copy())
            stream.consume(len(a)); count-=len(a)
    return np.concatenate(sites),np.concatenate(uniforms)
for a,b in zip(collect([83]),collect([1,7,25,50])):
    np.testing.assert_array_equal(a,b)
model=IsingModel(N=N,beta=.55,lam=2)
a=Simulator(model,seed=seed,backend='python')
b=Simulator(model,seed=seed,backend='python')
a.advance_updates(83)
for count in (1,7,25,50): b.advance_updates(count)
np.testing.assert_array_equal(a.state,b.state)
assert a.observe()==b.observe()
print('Chunk invariance verified.')

## Timing slicing versus copying
This microbenchmark illustrates array operations only. It does not establish that an integer cursor would improve simulator performance.

In [ ]:
import timeit
buffer=np.arange(100_000)
for statement in ('buffer[1:]','buffer[1:].copy()'):
    times=timeit.repeat(statement,globals={'buffer':buffer},number=1000,repeat=3)
    print(statement, min(times)/1000*1e6,'microseconds per call')
print(inspect.getsource(RandomUpdateStream))